# 19 Apply OTM Review Decisions

Apply the filled manual review decisions to the automatic OTM base and produce a cleaner final POI table.

In [1]:
from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)


In [2]:
BASE_PATH = "../data/processed/otm_pois_base.csv"
REVIEW_RAW_PATH = "../data/processed/otm_pois_base_review.csv"
REVIEW_FILLED_PATH = "/Users/imrandurmus/Downloads/otm_pois_base_review_filled.xlsx"

OUTPUT_FINAL_PATH = "../data/processed/otm_pois_base_final.csv"
OUTPUT_DISCUSSION_PATH = "../data/processed/otm_pois_base_discussion.csv"

base_df = pd.read_csv(BASE_PATH)
review_raw_df = pd.read_csv(REVIEW_RAW_PATH)

print("base_df:", base_df.shape)
print("review_raw_df:", review_raw_df.shape)


base_df: (327, 18)
review_raw_df: (104, 34)


## Read filled review workbook

In [3]:
def read_simple_xlsx(path, sheet_index=0):
    ns = {
        "a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
    }

    with ZipFile(path) as zf:
        shared_strings = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in root.findall("a:si", ns):
                text = "".join(t.text or "" for t in si.iter("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t"))
                shared_strings.append(text)

        wb = ET.fromstring(zf.read("xl/workbook.xml"))
        rels = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
        rel_map = {rel.attrib["Id"]: rel.attrib["Target"] for rel in rels}
        sheets = wb.findall("a:sheets/a:sheet", ns)
        target = "xl/" + rel_map[sheets[sheet_index].attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]]
        ws = ET.fromstring(zf.read(target))

        rows = []
        for row in ws.findall("a:sheetData/a:row", ns):
            values = {}
            for cell in row.findall("a:c", ns):
                ref = cell.attrib["r"]
                col = "".join(ch for ch in ref if ch.isalpha())
                cell_type = cell.attrib.get("t")
                v = cell.find("a:v", ns)
                if v is None:
                    value = ""
                elif cell_type == "s":
                    value = shared_strings[int(v.text)]
                else:
                    value = v.text or ""
                values[col] = value
            rows.append(values)

    if not rows:
        return pd.DataFrame()

    max_col = max(max((ord(c[0]) - 64 for c in row.keys()), default=0) for row in rows)
    ordered_cols = [chr(ord("A") + i) for i in range(max_col)]
    matrix = [[row.get(col, "") for col in ordered_cols] for row in rows]
    header = matrix[0]
    body = matrix[1:]
    return pd.DataFrame(body, columns=header)


filled_review_df = read_simple_xlsx(REVIEW_FILLED_PATH)
filled_review_df.head(20)


,xid,name,query_area,wikidata,wikipedia_url,candidate_count_same_key,kinds,review_decision,canonical_name,notes
0,N5110344551,Kepenekci sinan camii,Suleymaniye,Q106223671,17,2,"religion,other_temples,interesting_places",needs_discussion,Kepenekçi Sinan,Mosque vs madrasa/category name; related but n...
1,W330348666,Category:Kepenekçi Sinan Madrasa,Suleymaniye,Q106223671,17,2,"religion,other_temples,interesting_places",needs_discussion,Kepenekçi Sinan,Mosque vs madrasa/category name; related but n...
2,W104061375,Zeynep Sultan Camii,Hagia Sophia / Basilica Cistern,Q1153320,17,2,"religion,mosques,interesting_places",merge,Zeynep Sultan Mosque,Same mosque
3,N5108843571,Zeynep Sultan Mosque,Hagia Sophia / Basilica Cistern,Q1153320,17,2,"religion,mosques,interesting_places",merge,Zeynep Sultan Mosque,Same mosque
4,N1908677124,Milion,Hagia Sophia / Basilica Cistern,Q1187329,17,2,"milestones,architecture,historic_architecture,...",merge,Milion,Same monument
5,N5108843107,Milion Stone,Hagia Sophia / Basilica Cistern,Q1187329,17,2,"milestones,architecture,historic_architecture,...",merge,Milion,Same monument
6,R1555271,Hagia Sophia Grand Mosque,Hagia Sophia / Basilica Cistern,Q12506,https://en.wikipedia.org/wiki/Hagia%20Sophia,2,"religion,mosques,churches,cultural,museums,int...",merge,Hagia Sophia,Same place
7,W109862851,The Hagia Sophia Grand Mosque,Hagia Sophia / Basilica Cistern,Q12506,https://en.wikipedia.org/wiki/Hagia%20Sophia,2,"religion,mosques,churches,cultural,museums,int...",merge,Hagia Sophia,Same place
8,W136456159,Fethiye Museum,Balat / Fener,Q1420984,17,2,"religion,mosques,churches,cultural,museums,int...",merge,Fethiye Mosque / Museum,Same historic building
9,N5243953326,Fethiye Camii,Balat / Fener,Q1420984,17,2,"religion,mosques,churches,cultural,museums,int...",merge,Fethiye Mosque / Museum,Same historic building


## Merge review decisions with raw review rows

In [4]:
filled_review_df = filled_review_df.rename(columns={
    "review_decision": "review_decision",
    "canonical_name": "canonical_name",
    "notes": "review_notes",
})

filled_review_df["xid"] = filled_review_df["xid"].astype(str)
review_raw_df["xid"] = review_raw_df["xid"].astype(str)

review_full_df = review_raw_df.merge(
    filled_review_df[["xid", "review_decision", "canonical_name", "review_notes"]],
    on="xid",
    how="left",
)

print(review_full_df[["review_decision"]].value_counts(dropna=False).to_string())
review_full_df[["xid", "name", "review_decision", "canonical_name", "review_notes"]].head(20)


review_decision 
merge               87
needs_discussion     9
keep_separate        8


,xid,name,review_decision,canonical_name,review_notes
0,N7215645385,The Blue Mosque,keep_separate,Blue Mosque,Different POIs: mosque vs tomb
1,R1555271,Hagia Sophia Grand Mosque,merge,Hagia Sophia,Same place
2,W109862851,The Hagia Sophia Grand Mosque,merge,Hagia Sophia,Same place
3,W103953125,Tomb of Sultan Ahmet,keep_separate,Tomb of Sultan Ahmed,Different POIs: mosque vs tomb
4,W326372295,Yıldız Palace,merge,Yıldız Palace,Same palace
5,N4477143891,Yıldız Palace,merge,Yıldız Palace,Same palace
6,N7294561685,Chora Mosque,merge,Chora Mosque / Kariye Museum,Same historic building
7,W32395058,Kariye Museum,merge,Chora Mosque / Kariye Museum,Same historic building
8,R7318154,Rumeli Fortress,merge,Rumeli Fortress,Same fortress
9,W102190099,Osman ağa Mosque,merge,Osman Ağa Mosque,Same mosque


## Resolve groups

In [5]:
reviewed_group_keys = set(review_full_df["dedupe_key"].dropna().unique())
untouched_base_df = base_df.loc[~base_df["otm_xid"].isin(review_full_df["xid"])].copy()

resolved_rows = []
discussion_rows = []

for dedupe_key, group in review_full_df.groupby("dedupe_key"):
    decisions = set(group["review_decision"].dropna().astype(str).str.strip())

    if len(decisions) != 1:
        discussion_rows.append(group.copy())
        continue

    decision = next(iter(decisions))

    if decision == "merge":
        chosen = group.sort_values(["rate", "dist_from_query_center_m"], ascending=[False, True]).iloc[0].copy()
        canonical_name = str(group["canonical_name"].dropna().iloc[0]).strip()
        if canonical_name:
            chosen["name"] = canonical_name
            chosen["display_name_en"] = canonical_name
        chosen["needs_review"] = 0
        chosen["candidate_count_same_key"] = len(group)
        resolved_rows.append(chosen)

    elif decision == "keep_separate":
        group_keep = group.copy()
        group_keep["needs_review"] = 0
        resolved_rows.append(group_keep)

    elif decision == "needs_discussion":
        discussion_rows.append(group.copy())

    else:
        discussion_rows.append(group.copy())

resolved_df = pd.concat(
    [df if isinstance(df, pd.DataFrame) else pd.DataFrame([df]) for df in resolved_rows],
    ignore_index=True,
) if resolved_rows else pd.DataFrame()

discussion_df = pd.concat(discussion_rows, ignore_index=True) if discussion_rows else pd.DataFrame()

print("Untouched base rows:", untouched_base_df.shape)
print("Resolved rows:", resolved_df.shape)
print("Discussion rows:", discussion_df.shape)


Untouched base rows: (283, 18)
Resolved rows: (44, 37)
Discussion rows: (9, 37)


## Build final outputs

In [6]:
base_columns = list(base_df.columns)

if not resolved_df.empty:
    resolved_df = resolved_df.rename(columns={
        "xid": "otm_xid",
        "preview_source": "preview_image",
    })
    for col in base_columns:
        if col not in resolved_df.columns:
            resolved_df[col] = pd.NA
    resolved_df = resolved_df[base_columns]

final_df = pd.concat([untouched_base_df, resolved_df], ignore_index=True)
final_df = final_df.sort_values(["rate", "display_name_en"], ascending=[False, True]).reset_index(drop=True)

discussion_keep_cols = [c for c in [
    "xid", "name", "query_area", "wikidata", "wikipedia_url", "candidate_count_same_key", "review_decision", "canonical_name", "review_notes", "kinds"
] if c in discussion_df.columns]
discussion_export_df = discussion_df[discussion_keep_cols].copy() if not discussion_df.empty else discussion_df.copy()

print("Final base shape:", final_df.shape)
print(final_df["category_clean"].value_counts(dropna=False).to_string())
print("Discussion export shape:", discussion_export_df.shape)

final_df.head(30)


Final base shape: (327, 18)
category_clean
religious     139
historic       95
museum         49
attraction     44
Discussion export shape: (9, 10)


,poi_id,otm_xid,name,display_name_en,kinds,category_clean,wikidata,wikipedia_url,lat,lon,query_area,dist_from_query_center_m,rate,address_raw,preview_image,source,needs_review,candidate_count_same_key
0,otm_N7294561685,N7294561685,Chora Mosque / Kariye Museum,Chora Mosque / Kariye Museum,"religion,mosques,churches,cultural,museums,int...",museum,Q849489,https://en.wikipedia.org/wiki/Chora%20Church,41.031097,28.939091,Balat / Fener,905.679630,7,"{'city': 'Derviş Ali Mahallesi', 'road': 'Kari...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,2
1,otm_R1555271,R1555271,Hagia Sophia,Hagia Sophia,"religion,mosques,churches,cultural,museums,int...",museum,Q12506,https://en.wikipedia.org/wiki/Hagia%20Sophia,41.008507,28.980009,Hagia Sophia / Basilica Cistern,57.237152,7,"{'city': 'Cankurtaran Mahallesi', 'road': 'Bab...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,2
2,otm_N415157636,N415157636,Serpent Column,Serpent Column,"historic,monuments_and_memorials,burial_places...",historic,Q588892,https://en.wikipedia.org/wiki/Serpent%20Column,41.005661,28.975103,Sultanahmet Core,145.719100,7,"{'city': 'Binbirdirek Mahallesi', 'town': 'Fat...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,1
3,otm_R1564032,R1564032,Süleymaniye Mosque,Süleymaniye Mosque,"religion,mosques,interesting_places",religious,Q178643,https://en.wikipedia.org/wiki/S%C3%BCleymaniye...,41.016300,28.963978,Suleymaniye,18.644092,7,"{'city': 'Süleymaniye Mahallesi', 'road': 'Pro...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,1
4,otm_N7215645385,N7215645385,The Blue Mosque,The Blue Mosque,"religion,mosques,interesting_places",religious,Q80541,https://en.wikipedia.org/wiki/Sultan%20Ahmed%2...,41.005253,28.976892,Sultanahmet Core,18.136162,7,"{'town': 'Fatih', 'house': 'Sultanahmet Camii'...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,2
5,otm_W103953125,W103953125,Tomb of Sultan Ahmet,Tomb of Sultan Ahmet,"religion,mosques,interesting_places",religious,Q80541,https://en.wikipedia.org/wiki/Sultan%20Ahmed%2...,41.006790,28.977037,Sultanahmet Core,155.641087,7,"{'town': 'Fatih', 'state': 'Marmara Bölgesi', ...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,2
6,otm_W326372295,W326372295,Yıldız Palace,Yıldız Palace,"palaces,architecture,historic_architecture,cul...",museum,Q911734,https://en.wikipedia.org/wiki/Y%C4%B1ld%C4%B1z...,41.050076,29.011618,Yildiz Palace,700.923972,7,"{'city': 'Yıldız Mahallesi', 'road': 'Serenceb...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,2
7,otm_N6879165718,N6879165718,15 July coup monument (Istanbul),15 July coup monument (Istanbul),"historic,monuments_and_memorials,interesting_p...",historic,Q108404641,NaN,41.036991,29.042274,Beylerbeyi Palace,576.066703,3,NaN,NaN,opentripmap,0,1
8,otm_Q6063427,Q6063427,Abbas Ağa Fountain,Abbas Ağa Fountain,"fountains,historic,cultural,urban_environment,...",historic,Q6063427,NaN,41.021812,29.018139,Uskudar Waterfront,507.259033,3,NaN,NaN,opentripmap,0,1
9,otm_N3580821812,N3580821812,Adam Mickiewicz Museum,Adam Mickiewicz Museum,"biographical_museums,historic_house_museums,cu...",museum,Q4679495,NaN,41.038780,28.977148,Istiklal / Pera,545.861319,3,NaN,NaN,opentripmap,0,2


## Save outputs

In [7]:
final_df.to_csv(OUTPUT_FINAL_PATH, index=False)
discussion_export_df.to_csv(OUTPUT_DISCUSSION_PATH, index=False)

print("Saved:", OUTPUT_FINAL_PATH, final_df.shape)
print("Saved:", OUTPUT_DISCUSSION_PATH, discussion_export_df.shape)


Saved: ../data/processed/otm_pois_base_final.csv (327, 18)
Saved: ../data/processed/otm_pois_base_discussion.csv (9, 10)
